In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

raw_dir = Path('./raw_data')
clean_dir = Path('./jd_clean_output')
clean_dir.mkdir(parents=True, exist_ok=True)

In [2]:
skus = pd.read_csv(raw_dir / 'JD_sku_data.csv')
users = pd.read_csv(raw_dir / 'JD_user_data.csv')
clicks = pd.read_csv(raw_dir / 'JD_click_data.csv')
orders = pd.read_csv(raw_dir / 'JD_order_data.csv')
delivery = pd.read_csv(raw_dir / 'JD_delivery_data.csv')
inventory = pd.read_csv(raw_dir / 'JD_inventory_data.csv')
network = pd.read_csv(raw_dir / 'JD_network_data.csv')

for name, df in {
    'skus': skus,
    'users': users,
    'clicks': clicks,
    'orders': orders,
    'delivery': delivery,
    'inventory': inventory,
    'network': network
}.items():
    print(name, df.shape)

skus (31868, 7)
users (457298, 10)
clicks (6114124, 4)
orders (549989, 17)
delivery (293229, 6)
inventory (136079, 3)
network (56, 2)


In [3]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower().replace(' ', '_') for c in df.columns]
    return df

def to_string_id(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype('string').str.strip()
    return df

def to_datetime_cols(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce')
    return df

def missing_profile(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        'column': df.columns,
        'dtype': [str(x) for x in df.dtypes],
        'n_missing': df.isna().sum().values,
        'missing_rate': df.isna().mean().values,
        'n_unique': [df[c].nunique(dropna=True) for c in df.columns]
    })
    return out.sort_values(['missing_rate', 'n_missing'], ascending=False)

def uniqueness_check(df: pd.DataFrame, keys: list[str], name: str) -> pd.DataFrame:
    tmp = df.groupby(keys, dropna=False).size().reset_index(name='cnt')
    dup = tmp[tmp['cnt'] > 1].sort_values('cnt', ascending=False)
    print(f'[{name}] key={keys}, duplicate key rows =', len(dup))
    return dup

def safe_div(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

def mode_or_first(s: pd.Series):
    s2 = s.dropna()
    if len(s2) == 0:
        return np.nan
    m = s2.mode()
    return m.iloc[0] if len(m) > 0 else s2.iloc[0]

In [4]:
skus_ = clean_columns(skus)
users_ = clean_columns(users)
clicks_ = clean_columns(clicks)
orders_ = clean_columns(orders)
delivery_ = clean_columns(delivery)
inventory_ = clean_columns(inventory)
network_ = clean_columns(network)

skus_ = to_string_id(skus_, ['sku_id', 'brand_id'])
users_ = to_string_id(users_, ['user_id'])
clicks_ = to_string_id(clicks_, ['user_id', 'sku_id', 'channel'])
orders_ = to_string_id(orders_, ['order_id', 'user_id', 'sku_id'])
delivery_ = to_string_id(delivery_, ['package_id', 'order_id'])
inventory_ = to_string_id(inventory_, ['sku_id'])

skus_ = to_datetime_cols(skus_, ['activate_date', 'deactivate_date'])
clicks_ = to_datetime_cols(clicks_, ['request_time'])
orders_ = to_datetime_cols(orders_, ['order_time', 'order_date'])
delivery_ = to_datetime_cols(delivery_, ['ship_out_time', 'arr_station_time', 'arr_time'])
inventory_ = to_datetime_cols(inventory_, ['date'])

In [5]:
dup_skus = uniqueness_check(skus_, ['sku_id'], 'skus')
dup_users = uniqueness_check(users_, ['user_id'], 'users')
dup_inventory = uniqueness_check(inventory_, ['dc_id', 'sku_id', 'date'], 'inventory')
dup_network = uniqueness_check(network_, ['region_id', 'dc_id'], 'network')

print('\norders 行数:', len(orders_))
print('orders 唯一 order_id 数:', orders_['order_id'].nunique(dropna=True))
print('说明: orders 不是一行一个订单，而是一行一个订单行 item.')

print('\ndelivery 行数:', len(delivery_))
print('delivery 唯一 package_id 数:', delivery_['package_id'].nunique(dropna=True))
print('delivery 唯一 order_id 数:', delivery_['order_id'].nunique(dropna=True))
print('说明: 一个订单可能拆成多个 package.')

[skus] key=['sku_id'], duplicate key rows = 1
[users] key=['user_id'], duplicate key rows = 0
[inventory] key=['dc_id', 'sku_id', 'date'], duplicate key rows = 0
[network] key=['region_id', 'dc_id'], duplicate key rows = 0

orders 行数: 549989
orders 唯一 order_id 数: 486928
说明: orders 不是一行一个订单，而是一行一个订单行 item.

delivery 行数: 293229
delivery 唯一 package_id 数: 293229
delivery 唯一 order_id 数: 293204
说明: 一个订单可能拆成多个 package.


In [6]:
audit_tables = {
    'skus': skus_,
    'users': users_,
    'clicks': clicks_,
    'orders': orders_,
    'delivery': delivery_,
    'inventory': inventory_,
    'network': network_
}

for name, df in audit_tables.items():
    print('\n' + '#' * 80)
    print(name)
    display(missing_profile(df).head(20))


################################################################################
skus


,column,dtype,n_missing,missing_rate,n_unique
6,deactivate_date,datetime64[us],30727,0.964196,31
5,activate_date,datetime64[us],28810,0.904042,31
0,sku_id,string,0,0.000000,31867
1,type,int64,0,0.000000,2
2,brand_id,string,0,0.000000,1890
3,attribute1,str,0,0.000000,5
4,attribute2,str,0,0.000000,9



################################################################################
users


,column,dtype,n_missing,missing_rate,n_unique
0,user_id,string,0,0.0,457298
1,user_level,int64,0,0.0,7
2,first_order_month,str,0,0.0,169
3,plus,int64,0,0.0,2
4,gender,str,0,0.0,3
5,age,str,0,0.0,7
6,marital_status,str,0,0.0,3
7,education,int64,0,0.0,5
8,city_level,int64,0,0.0,6
9,purchase_power,int64,0,0.0,6



################################################################################
clicks


...

,column,dtype,n_missing,missing_rate,n_unique
0,package_id,string,0,0.0,293229
1,order_id,string,0,0.0,293204
2,type,int64,0,0.0,2
3,ship_out_time,datetime64[us],0,0.0,760
4,arr_station_time,datetime64[us],0,0.0,832
5,arr_time,datetime64[us],0,0.0,695



################################################################################
inventory


,column,dtype,n_missing,missing_rate,n_unique
0,dc_id,int64,0,0.0,56
1,sku_id,string,0,0.0,390
2,date,datetime64[us],0,0.0,31



################################################################################
network


,column,dtype,n_missing,missing_rate,n_unique
0,region_id,int64,0,0.0,8
1,dc_id,int64,0,0.0,56


In [7]:
discount_cols = [
    'direct_discount_per_unit',
    'quantity_discount_per_unit',
    'bundle_discount_per_unit',
    'coupon_discount_per_unit'
]

for c in discount_cols:
    if c not in orders_.columns:
        orders_[c] = 0.0

orders_['discount_gap_check'] = (
    orders_['original_unit_price'].fillna(0)
    - orders_['final_unit_price'].fillna(0)
    - orders_['direct_discount_per_unit'].fillna(0)
    - orders_['quantity_discount_per_unit'].fillna(0)
    - orders_['bundle_discount_per_unit'].fillna(0)
    - orders_['coupon_discount_per_unit'].fillna(0)
)

orders_['discount_gap_abs'] = orders_['discount_gap_check'].abs()
display(orders_['discount_gap_abs'].describe())
print('折扣对账异常比例(>1e-6):', (orders_['discount_gap_abs'] > 1e-6).mean())

first_click = clicks_.groupby(['user_id', 'sku_id'], as_index=False).agg(first_click_time=('request_time', 'min'))
first_order = orders_.groupby(['user_id', 'sku_id'], as_index=False).agg(first_order_time=('order_time', 'min'))

click_order_check = first_order.merge(first_click, on=['user_id', 'sku_id'], how='left')
click_order_check['order_before_click'] = click_order_check['first_order_time'] < click_order_check['first_click_time']
print('订单早于点击的比例（仅有 click 记录的 user-sku）:',
      click_order_check['order_before_click'].fillna(False).mean())

known_click_user_rate = clicks_['user_id'].isin(users_['user_id']).mean()
print('clicks 中 user_id 被 users 覆盖比例:', known_click_user_rate)

delivery_cover_rate = orders_['order_id'].isin(delivery_['order_id']).mean()
print('orders 行被 delivery 匹配的比例:', delivery_cover_rate)
print('unique order_id 被 delivery 匹配的比例:',
      orders_['order_id'].dropna().isin(delivery_['order_id']).mean())

count    5.499890e+05
mean     2.889851e-16
std      1.396136e-15
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      4.729550e-14
Name: discount_gap_abs, dtype: float64

折扣对账异常比例(>1e-6): 0.0
订单早于点击的比例（仅有 click 记录的 user-sku）: 0.014137824912252215
clicks 中 user_id 被 users 覆盖比例: 0.33309465100805935
orders 行被 delivery 匹配的比例: 0.5942555214740658
unique order_id 被 delivery 匹配的比例: 0.5942555214740658


In [8]:
skus_.to_csv(clean_dir / 'skus_clean.csv', index=False)
users_.to_csv(clean_dir / 'users_clean.csv', index=False)
clicks_.to_csv(clean_dir / 'clicks_clean.csv', index=False)
orders_.to_csv(clean_dir / 'orders_clean.csv', index=False)
delivery_.to_csv(clean_dir / 'delivery_clean.csv', index=False)
inventory_.to_csv(clean_dir / 'inventory_clean.csv', index=False)
network_.to_csv(clean_dir / 'network_clean.csv', index=False)

print('已导出到:', clean_dir.resolve())

已导出到: /home/houga/JD/jd_clean_output
